# Day 32 PM – Insurance Fraud Detection: Decision Tree & Random Forest

Synthetic insurance claims dataset with DT, tuned RF, cost-sensitive evaluation, and Gradient Boosting preview.

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              recall_score, precision_score, confusion_matrix)


## Part A – Fraud Detection: Decision Tree and Random Forest

In [ ]:
rng = np.random.default_rng(42)
n = 3000

claim_amount = rng.normal(50000, 20000, n).clip(5000, 200000)
num_claims_past = rng.integers(0, 11, n)
days_since_policy = rng.integers(1, 3651, n)
num_witnesses = rng.integers(0, 6, n)
claim_hour = rng.integers(0, 24, n)
vehicle_age = rng.integers(0, 21, n)
deductible_ratio = rng.uniform(0.05, 0.5, n)
policy_premium = rng.normal(15000, 5000, n).clip(3000, 50000)

fraud_score = (
    0.000005 * (claim_amount - 40000)
    + 0.4 * (num_claims_past - 2)
    - 0.0001 * (days_since_policy - 500)
    - 0.3 * (num_witnesses - 1)
    + 0.05 * (claim_hour < 6).astype(int) * 2
    + 0.1 * (vehicle_age > 15).astype(int)
    - 0.5 * deductible_ratio
)

noise = rng.normal(0, 0.5, n)
prob = 1 / (1 + np.exp(-(fraud_score + noise)))
fraud = (prob > 0.55).astype(int)

df = pd.DataFrame({
    "claim_amount": claim_amount.round(0),
    "num_claims_past": num_claims_past,
    "days_since_policy": days_since_policy,
    "num_witnesses": num_witnesses,
    "claim_hour": claim_hour,
    "vehicle_age": vehicle_age,
    "deductible_ratio": deductible_ratio.round(3),
    "policy_premium": policy_premium.round(0),
    "fraud": fraud
})

print(df.head())
print(df["fraud"].value_counts(normalize=True).round(3))
print("Fraud rate:", df["fraud"].mean().round(3))


In [ ]:
X = df.drop(columns=["fraud"])
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
proba_dt = dt.predict_proba(X_test)[:, 1]

acc_dt  = accuracy_score(y_test, y_pred_dt)
f1_dt   = f1_score(y_test, y_pred_dt)
rec_dt  = recall_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
roc_dt  = roc_auc_score(y_test, proba_dt)

print("Decision Tree")
print(f"Accuracy : {acc_dt:.4f}")
print(f"Precision: {prec_dt:.4f}")
print(f"Recall   : {rec_dt:.4f}")
print(f"F1       : {f1_dt:.4f}")
print(f"ROC-AUC  : {roc_dt:.4f}")


In [ ]:
feature_names = list(X.columns)

leaf_indices = np.where(dt.tree_.children_left == -1)[0]

rules = []
for leaf in leaf_indices:
    path = []
    node = leaf
    while node != 0:
        parents = np.where(
            (dt.tree_.children_left == node) | (dt.tree_.children_right == node)
        )[0]
        parent = int(parents[0])
        is_left = dt.tree_.children_left[parent] == node
        col = feature_names[dt.tree_.feature[parent]]
        thr = dt.tree_.threshold[parent]
        path.append((col, "<=" if is_left else ">", thr))
        node = parent
    path = path[::-1]

    mask = np.ones(len(X), dtype=bool)
    for col, op, thr in path:
        if op == "<=":
            mask &= (X[col].values <= thr)
        else:
            mask &= (X[col].values > thr)

    subset = y[mask]
    if len(subset) == 0:
        continue
    majority = int(subset.value_counts().idxmax())
    acc_rule = (subset == majority).mean()
    rules.append({"path": path, "majority": majority, "acc": acc_rule, "count": len(subset)})

top3 = sorted(rules, key=lambda r: r["count"], reverse=True)[:3]

for i, r in enumerate(top3, 1):
    cond = " AND ".join(f"{c} {o} {t:.2f}" for c, o, t in r["path"])
    label = "FRAUD" if r["majority"] == 1 else "NOT FRAUD"
    print(f"Rule {i}: IF {cond} -> {label}  ({r['acc']:.2f} acc, {r['count']} samples)")


In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [None, 6, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [{0: 1, 1: 10}, "balanced", None]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True)

search = RandomizedSearchCV(
    rf_base, param_distributions=param_dist,
    n_iter=20, cv=5, scoring="recall",
    n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)

rf = search.best_estimator_

y_pred_rf = rf.predict(X_test)
proba_rf  = rf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
rec_rf  = recall_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
roc_rf  = roc_auc_score(y_test, proba_rf)

print("Best params:", search.best_params_)
print(f"
OOB Score: {rf.oob_score_:.4f}")
print("
Random Forest")
print(f"Accuracy : {acc_rf:.4f}")
print(f"Precision: {prec_rf:.4f}")
print(f"Recall   : {rec_rf:.4f}")
print(f"F1       : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_rf:.4f}")


In [ ]:
metrics_df = pd.DataFrame({
    "Model":     ["Decision Tree (depth=5)", "Random Forest (tuned)"],
    "Accuracy":  [round(acc_dt, 4), round(acc_rf, 4)],
    "Precision": [round(prec_dt, 4), round(prec_rf, 4)],
    "Recall":    [round(rec_dt, 4), round(rec_rf, 4)],
    "F1":        [round(f1_dt, 4), round(f1_rf, 4)],
    "ROC-AUC":   [round(roc_dt, 4), round(roc_rf, 4)],
})
print(metrics_df.to_string(index=False))


In [ ]:
FP_COST = 1
FN_COST = 10

def total_cost(y_true, y_pred, fp_cost=FP_COST, fn_cost=FN_COST):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return fp * fp_cost + fn * fn_cost

cost_dt = total_cost(y_test, y_pred_dt)
cost_rf = total_cost(y_test, y_pred_rf)

print("Cost-Sensitive Evaluation (FN cost = 10x FP cost)")
print(f"Decision Tree total cost : {cost_dt}")
print(f"Random Forest total cost : {cost_rf}")
print(f"RF saves {cost_dt - cost_rf} units of cost vs Decision Tree")


## Deployment Recommendation

**Paragraph 1:**
The Random Forest model should be deployed as the primary automated fraud scoring engine because it achieves significantly higher recall and lower total cost under the 10x FN penalty.
Fraud that goes undetected is ten times more expensive than a false alarm that triggers a manual review, so optimizing for recall is the correct business objective.
RandomizedSearchCV with recall scoring was used to tune the Random Forest, and the `class_weight` parameter was included in the search grid to let the model account for the class imbalance directly.

**Paragraph 2:**
The Decision Tree with max_depth=5 should be retained as the regulatory explanation layer.
Regulators require interpretable model outputs, and the top 3 extracted rules provide clear, auditable logic that investigators and compliance teams can understand without needing data science expertise.
A practical deployment pipeline would run the Random Forest to score every incoming claim and then surface the relevant Decision Tree rule for any claim flagged as high-risk, satisfying both accuracy and regulatory requirements simultaneously.


## Part B – Gradient Boosting Preview (Self-Study)

In [ ]:
boosting_vs_bagging = {
    "Bagging (Random Forest)": [
        "Trains trees independently in parallel on bootstrapped samples",
        "Averages predictions to reduce variance",
        "Each tree has equal weight in final vote",
        "Good when individual trees overfit (high variance)"
    ],
    "Boosting (Gradient Boosting)": [
        "Trains trees sequentially, each correcting the previous tree's errors",
        "Each tree focuses on residuals (mistakes) of the ensemble so far",
        "Trees are weighted by their contribution to error reduction",
        "Good when individual trees underfit (high bias)"
    ]
}

for method, points in boosting_vs_bagging.items():
    print(method)
    for p in points:
        print(" -", p)
    print()

boosting_resource = {
    "title": "Gradient Boosting explained (Towards Data Science)",
    "url": "https://towardsdatascience.com/understanding-gradient-boosting-machines-9be756fe76ab"
}
print("Recommended Resource:")
print(boosting_resource["title"])
print(boosting_resource["url"])
